### Exploración del dataset

In [9]:
# Importación de libererías necesarias para el análisis de datos
import pandas as pd
import numpy as np
import os
from pathlib import Path


In [10]:
# Ajusta esta ruta si tus CSV están en otra carpeta
# DATA_DIR = Path(".")   # carpeta actual por defecto
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path(".").resolve().parent
DATA_RAW = ROOT / "data" / "raw"
DATA_PRO = ROOT / "data" / "processed"
#DATA_DIR = Path("recommender-system/data/raw")  # o esta si ya tienes la estructura del proyecto

In [11]:
SEP = "=" * 60

def section(title):
    print(f"\n{SEP}")
    print(f"  {title}")
    print(SEP)

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. EVENTS.CSV
# ─────────────────────────────────────────────────────────────────────────────
section("1. EVENTS.CSV")

events = pd.read_csv(DATA_RAW / "events.csv")
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")

print(f"\nShape: {events.shape}")
print(f"\nDtypes:\n{events.dtypes}")
print(f"\nPrimeras 5 filas:\n{events.head()}")
print(f"\nNulls por columna:\n{events.isnull().sum()}")

print(f"\n--- Distribución de eventos ---")
print(events["event"].value_counts())
print(events["event"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print(f"\n--- Estadísticas generales ---")
print(f"Visitantes únicos:      {events['visitorid'].nunique():,}")
print(f"Ítems únicos:           {events['itemid'].nunique():,}")
print(f"Transacciones únicas:   {events['transactionid'].nunique():,}")
print(f"Fecha inicio:           {events['datetime'].min()}")
print(f"Fecha fin:              {events['datetime'].max()}")
print(f"Período total (días):   {(events['datetime'].max() - events['datetime'].min()).days}")

print(f"\n--- Duplicados exactos ---")
dupes = events.duplicated(subset=["timestamp","visitorid","event","itemid"]).sum()
print(f"Duplicados: {dupes:,}")

print(f"\n--- TransactionID en eventos 'transaction' ---")
trans_events = events[events["event"] == "transaction"]
print(f"Total eventos transaction:         {len(trans_events):,}")
print(f"Con transactionid nulo:            {trans_events['transactionid'].isna().sum():,}")
print(f"Con transactionid vacío (''):      {(trans_events['transactionid'] == '').sum():,}")
print(f"Transacciones válidas:             {trans_events['transactionid'].notna().sum():,}")

print(f"\n--- Interacciones por usuario (top estadísticas) ---")
user_counts = events.groupby("visitorid")["event"].count()
print(user_counts.describe())
print(f"\nUsuarios con solo 1 evento:  {(user_counts == 1).sum():,}")
print(f"Usuarios con 2 eventos:      {(user_counts == 2).sum():,}")
print(f"Usuarios con >= 3 eventos:   {(user_counts >= 3).sum():,}")
print(f"Usuarios con >= 10 eventos:  {(user_counts >= 10).sum():,}")

print(f"\n--- Top 10 ítems más vistos ---")
print(events[events["event"]=="view"].groupby("itemid")["event"].count()
      .sort_values(ascending=False).head(10))

print(f"\n--- Eventos por día de la semana ---")
events["dow"] = events["datetime"].dt.day_name()
print(events.groupby("dow")["event"].count().sort_values(ascending=False))



  1. EVENTS.CSV

Shape: (2756101, 6)

Dtypes:
timestamp                 int64
visitorid                 int64
event                    object
itemid                    int64
transactionid           float64
datetime         datetime64[ns]
dtype: object

Primeras 5 filas:
       timestamp  visitorid event  itemid  transactionid  \
0  1433221332117     257597  view  355908            NaN   
1  1433224214164     992329  view  248676            NaN   
2  1433221999827     111016  view  318965            NaN   
3  1433221955914     483717  view  253185            NaN   
4  1433221337106     951259  view  367447            NaN   

                 datetime  
0 2015-06-02 05:02:12.117  
1 2015-06-02 05:50:14.164  
2 2015-06-02 05:13:19.827  
3 2015-06-02 05:12:35.914  
4 2015-06-02 05:02:17.106  

Nulls por columna:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
datetime               0
dtype: int64

--- Distribució

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. ITEM_PROPERTIES
# ─────────────────────────────────────────────────────────────────────────────
section("2. ITEM_PROPERTIES (parte 1 y 2)")

dfs = []
for fname in ["item_properties_part1.csv", "item_properties_part2.csv"]:
    fpath = DATA_RAW / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f"\n{fname}: {df.shape}")
        dfs.append(df)
    else:
        print(f"ARCHIVO NO ENCONTRADO: {fname}")

if dfs:
    props = pd.concat(dfs, ignore_index=True)
    print(f"\nShape combinado:        {props.shape}")
    print(f"\nDtypes:\n{props.dtypes}")
    print(f"\nPrimeras 5 filas:\n{props.head()}")
    print(f"\nNulls:\n{props.isnull().sum()}")
    print(f"\nÍtems únicos:           {props['itemid'].nunique():,}")
    print(f"Propiedades únicas:     {props['property'].nunique():,}")

    print(f"\n--- Top 20 propiedades más frecuentes ---")
    print(props["property"].value_counts().head(20))

    print(f"\n--- Muestra de valores de 'categoryid' ---")
    cat_vals = props[props["property"]=="categoryid"]["value"].dropna()
    print(f"Valores únicos de categoryid: {cat_vals.nunique():,}")
    print(cat_vals.head(10).tolist())

    print(f"\n--- Detección de valores hasheados ---")
    props["val_len"] = props["value"].astype(str).str.len()
    print(props["val_len"].describe())
    print(f"\nValores con longitud > 20 chars (probablemente hasheados): "
        f"{(props['val_len'] > 20).sum():,} "
        f"({(props['val_len'] > 20).mean()*100:.1f}%)")

    print(f"\n--- Ítems con propiedades legibles vs hasheadas ---")
    readable = props[props["val_len"] <= 20]["itemid"].nunique()
    hashed   = props[props["val_len"] > 20]["itemid"].nunique()
    print(f"Ítems con al menos 1 valor legible:  {readable:,}")
    print(f"Ítems con al menos 1 valor hasheado: {hashed:,}")



  2. ITEM_PROPERTIES (parte 1 y 2)

item_properties_part1.csv: (10999999, 4)

item_properties_part2.csv: (9275903, 4)

Shape combinado:        (20275902, 4)

Dtypes:
timestamp     int64
itemid        int64
property     object
value        object
dtype: object

Primeras 5 filas:
       timestamp  itemid    property                            value
0  1435460400000  460429  categoryid                             1338
1  1441508400000  206783         888          1116713 960601 n277.200
2  1439089200000  395014         400  n552.000 639502 n720.000 424566
3  1431226800000   59481         790                       n15360.000
4  1431831600000  156781         917                           828513

Nulls:
timestamp    0
itemid       0
property     0
value        0
dtype: int64

Ítems únicos:           417,053
Propiedades únicas:     1,104

--- Top 20 propiedades más frecuentes ---
property
888           3000398
790           1790516
available     1503639
categoryid     788214
6              6

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. CATEGORY_TREE.CSV
# ─────────────────────────────────────────────────────────────────────────────
section("3. CATEGORY_TREE.CSV")

cat_tree = pd.read_csv(DATA_RAW / "category_tree.csv")
print(f"\nShape: {cat_tree.shape}")
print(f"\nDtypes:\n{cat_tree.dtypes}")
print(f"\nPrimeras 10 filas:\n{cat_tree.head(10)}")
print(f"\nNulls:\n{cat_tree.isnull().sum()}")
print(f"\nCategorías raíz (sin parentid): {cat_tree['parentid'].isna().sum():,}")
print(f"Categorías con padre:            {cat_tree['parentid'].notna().sum():,}")

# Profundidad del árbol
print(f"\n--- Muestra del árbol jerárquico ---")
roots = cat_tree[cat_tree["parentid"].isna()]["categoryid"].tolist()
print(f"Categorías raíz: {roots[:10]}")
children_of_first = cat_tree[cat_tree["parentid"] == roots[0]]["categoryid"].tolist() if roots else []
print(f"Hijos de la primera raíz ({roots[0] if roots else 'N/A'}): {children_of_first[:10]}")



  3. CATEGORY_TREE.CSV

Shape: (1669, 2)

Dtypes:
categoryid      int64
parentid      float64
dtype: object

Primeras 10 filas:
   categoryid  parentid
0        1016     213.0
1         809     169.0
2         570       9.0
3        1691     885.0
4         536    1691.0
5         231       NaN
6         542     378.0
7        1146     542.0
8        1140     542.0
9        1479    1537.0

Nulls:
categoryid     0
parentid      25
dtype: int64

Categorías raíz (sin parentid): 25
Categorías con padre:            1,644

--- Muestra del árbol jerárquico ---
Categorías raíz: [231, 791, 1490, 431, 755, 378, 1579, 1394, 659, 1057]
Hijos de la primera raíz (231): []


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. RESUMEN EJECUTIVO
# ─────────────────────────────────────────────────────────────────────────────
section("4. RESUMEN EJECUTIVO — MÉTRICAS CLAVE")

if dfs:
    items_with_props = props["itemid"].nunique()
    items_in_events  = events["itemid"].nunique()
    items_overlap    = len(set(props["itemid"]) & set(events["itemid"]))

    print(f"\nÍtems en events:            {items_in_events:,}")
    print(f"Ítems en item_properties:   {items_with_props:,}")
    print(f"Ítems en ambos (overlap):   {items_overlap:,}")
    print(f"Ítems solo en events:       {items_in_events - items_overlap:,}")
    print(f"Ítems solo en properties:   {items_with_props - items_overlap:,}")

print(f"\n--- Funnel de conversión ---")
for evt, cnt in events["event"].value_counts().items():
    pct = cnt / len(events) * 100
    print(f"  {evt:<15}: {cnt:>10,}  ({pct:.2f}%)")

views = events[events["event"]=="view"]["visitorid"].nunique()
carts = events[events["event"]=="addtocart"]["visitorid"].nunique()
buys  = events[events["event"]=="transaction"]["visitorid"].nunique()
total = events["visitorid"].nunique()

print(f"\n--- Funnel por usuarios únicos ---")
print(f"  Visitantes totales:     {total:,}  (100%)")
print(f"  Llegaron a view:        {views:,}  ({views/total*100:.1f}%)")
print(f"  Llegaron a addtocart:   {carts:,}  ({carts/total*100:.1f}%)")
print(f"  Llegaron a transaction: {buys:,}   ({buys/total*100:.2f}%)")




  4. RESUMEN EJECUTIVO — MÉTRICAS CLAVE

Ítems en events:            235,061
Ítems en item_properties:   417,053
Ítems en ambos (overlap):   185,246
Ítems solo en events:       49,815
Ítems solo en properties:   231,807

--- Funnel de conversión ---
  view           :  2,664,312  (96.67%)
  addtocart      :     69,332  (2.52%)
  transaction    :     22,457  (0.81%)

--- Funnel por usuarios únicos ---
  Visitantes totales:     1,407,580  (100%)
  Llegaron a view:        1,404,179  (99.8%)
  Llegaron a addtocart:   37,722  (2.7%)
  Llegaron a transaction: 11,719   (0.83%)
